# 📓 Notebook 1 — Fetch & Clean Tamil Corpus

> **Goal:** Collect raw text from all sources, strip non-Tamil content, and produce a single clean corpus file ready for tokenizer training.

***

## 📌 Design Decision: Tamil-Only Corpus

All data fed into the tokenizer training contains **only Tamil script characters** (Unicode block U+0B80–U+0BFF), Arabic numerals (0–9), Tamil numerals (௧–௯), and Tamil punctuation.

English words, Latin characters, XML/HTML markup, URLs, and wiki syntax are stripped before the corpus is assembled.

**Why?** The tokenizer learns its vocabulary from the training text. Every Latin character or English word that appears in training steals a vocabulary slot that could serve Tamil morphology. A tokenizer trained on pure Tamil text will achieve lower fertility scores (fewer subword splits per word) and higher coverage on real Tamil documents.

***

## 🔁 In Order to Expand Vocabulary to Multiple Languages

Expanding a trained WordPiece tokenizer to support additional languages is non-trivial. A WordPiece vocabulary is **not a neural network** — it is a deterministic lookup table built once by a merge algorithm. There is no checkpoint to resume, no gradient to continue. The options are:

- **Retrain from scratch on a multilingual corpus** — The cleanest approach. Combine Tamil text with the new language's text, re-run tokenizer training with a larger `vocab_size`, and rebuild the entire BERT model from scratch (since all token IDs shift). Expensive but produces an optimal joint vocabulary with correct frequency-weighted merges across both scripts.

- **Vocabulary extension (append new tokens)** — Keep the existing Tamil tokenizer's vocabulary and token IDs intact. Identify tokens in the new language that are absent from the current vocab, and append them using `tokenizer.add_tokens(new_tokens)` followed by `model.resize_token_embeddings(len(tokenizer))`. Existing Tamil token IDs are stable — only the new tokens need embedding initialisation (random or averaged from neighbours). Fine-tune on multilingual data to train the new embeddings. This avoids full retraining but produces a suboptimal vocabulary since new-language tokens were not part of the original frequency-weighted merge process.

- **Reserve `[unused]` slots at training time** — Pre-allocate spare vocabulary slots (e.g., `[unused0]` through `[unused999]`) during the original tokenizer training by setting `vocab_size` higher than strictly needed for Tamil. When expanding to a new language, fill those `[unused]` slots with the new script's tokens. Token IDs never shift, the embedding matrix never resizes, and the BERT model requires no architectural changes — only fine-tuning of the newly filled slots. This is the lowest-disruption expansion path and is exactly the strategy used by the original BERT-base model (which shipped with 994 `[unused]` tokens for this reason).

- **Use a script-aware subword model (SentencePiece BPE)** — Switch from WordPiece to SentencePiece with a BPE or Unigram model, which handles multilingual vocabularies more naturally and can be retrained with `--vocab_size` and `--input` pointing to a combined corpus. Not applicable to the current checkpoint, but worth noting for future architecture decisions if multilingual support becomes a first-class goal.

> **For TamBert:** Since this model is Tamil-first and multilingual expansion is a future concern, the current training uses a Tamil-only corpus and a pure Tamil vocabulary. If expansion is needed later, the **vocabulary extension path** is the lowest-cost option that preserves the existing checkpoint.

***

## Folder structure

```
tambert/
└── data/
    ├── raw/
    │   ├── tamil_wiki/
    │   ├── cc100/
    │   └── project_madurai/
    ├── cleaned/
    │   ├── tamil_wiki.txt
    │   ├── cc100.txt
    │   └── project_madurai.txt
    └── corpus/
        ├── corpus_90pct.txt   ← tokenizer + MLM training
        └── corpus_10pct.txt   ← held-out coverage check
```

## Wikipedia Data


### Corpus Cleaning & Tokenizer Design Decisions

1. TAMIL_RUN Evolution

**Original:** Captured only pure Tamil Unicode block runs (`[\u0B80-\u0BFF]+`).
Dropped everything else — numbers, punctuation, abbreviations.

**Problem discovered:** Tamil abbreviations like `ச.கி.மீ` (sq. km) were split
into three separate tokens because `.` broke the run. Numbers were also absent.

**Final version:**
```python
TAMIL_RUN = re.compile(
    r'[\u0B80-\u0BFF]+(?:\.[\u0B80-\u0BFF]+)*'  # Tamil + dot-abbreviations: ச.கி.மீ
    r'|[0-9]+(?:\.[0-9]+)?'                       # integers + decimals: 3.14
)

def extract_tamil_words(text: str) -> list[str]:
    text = ENGLISH_WORD.sub('', text)  # strip all Latin
    return TAMIL_RUN.findall(text)
```

2. Why All Latin (`[A-Za-z]+`) Was Removed

Tamil Wikipedia uses Tamil-script abbreviations (`கி.மீ`, not `km`). Any
surviving Latin characters are wikitext parser artifacts — not meaningful
prose. Removing all Latin is safe for this corpus.

3. Why Symbols (`+ = ( ) ; |` etc.) Were Excluded

Stray symbols in the output are remnants of `{{infobox}}` templates that
`clean_wikitext` partially strips. They carry no linguistic signal.
More importantly, vocabulary slots at small scale are better spent on
Tamil morphemes than punctuation.

4. Scope Decision: Tamil Fluency First

Math, coding, and mixed-script capability require symbols (`+`, `=`, `()`)
and a richer tokenizer. These are deliberately deferred. The path forward:

- Train v1 on clean Tamil-only corpus with an **oversized `vocab_size`**
- Reserve `[unused]` slots (as original BERT did with 994 slots)
- Extend tokenizer later via `add_tokens()` + `model.resize_token_embeddings()`
  without shifting any existing Tamil token IDs or retraining from scratch

In [1]:
! wget https://dumps.wikimedia.org/tawiki/latest/tawiki-latest-pages-articles.xml.bz2

--2026-06-27 23:02:12--  https://dumps.wikimedia.org/tawiki/latest/tawiki-latest-pages-articles.xml.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.242, 2620:0:861:ed1a::3:242
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.242|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 266603277 (254M) [application/octet-stream]
Saving to: ‘tawiki-latest-pages-articles.xml.bz2’

tawiki-latest-pages 100%[===================>] 254.25M  4.55MB/s    in 57s     

2026-06-27 23:03:09 (4.48 MB/s) - ‘tawiki-latest-pages-articles.xml.bz2’ saved [266603277/266603277]



In [2]:
from pathlib import Path
# Setup the data Path
data = Path('data')
data.mkdir(exist_ok=True, parents=True)
print('Data folder created...')

raw_data = data / Path('raw')
raw_data.mkdir(exist_ok=True, parents=True)
print('Raw folder created...')

Data folder created...
Raw folder created...


In [3]:
%%time

# Create folder for wiki exerpts
tamil_wiki = raw_data / Path('tamil_wiki')
tamil_wiki.mkdir(exist_ok=True, parents=True)

# Unzip the new format of compressed file
import bz2, shutil
out_path = tamil_wiki / 'tawiki-latest-pages-articles.xml'

with bz2.BZ2File('tawiki-latest-pages-articles.xml.bz2') as input, open(out_path, 'wb') as out:
    shutil.copyfileobj(input, out)

print('De-compression Complete...')

De-compression Complete...
CPU times: user 1min 29s, sys: 4.09 s, total: 1min 33s
Wall time: 1min 36s


### Extract the wikipedia data

The data is too big to be explored using pandas dataframe, so we would be using `lxml` sdk

#### Choosing vocabulary that the tokenizer would ideally learn

We keep intact selected list of punctuation so that their semantic meaning could be understood.

As this tokenizer would be used on a primarily literature based model, and does not specialize on math or coding this seemed sufficient

In [4]:
import re

# Matches any single character in the Tamil Unicode block (U+0B80–U+0BFF).
# This range covers Tamil letters, vowel signs, and digits.
ENGLISH_WORD  = re.compile(r'[A-Za-z]+')   # removes ALL Latin, including single chars
TAMIL_RUN = re.compile(
    r'[\u0B80-\u0BFF]+(?:\.[\u0B80-\u0BFF]+)*'  # Tamil + dot-abbreviations: கி.மீ
    r'|[0-9]+(?:\.[0-9]+)?'                       # integers + decimals: 3.14
    r'|[,!?"\-%\']'                           # prose punctuation — human-typed
    # = [ ] { } are intentionally excluded from training text.
    # Reserve them as [unused] slots via vocab_size overshoot.
    # Fill them later if you add math/code capability.
)

def extract_tamil_words(text: str) -> list[str]:
    text = ENGLISH_WORD.sub('', text)    # remove all English/Latin
    return TAMIL_RUN.findall(text)       # keep Tamil + symbols

#### Cleaning data using Regex

In [5]:
from lxml import etree

def clean_wikitext(text: str) -> str:
    """
    Strip MediaWiki markup down to plain prose.
    Wikitext is NOT plain text -- it's full of {{templates}}, [[links]],
    <ref> citations, HTML tags, etc. We need to remove the markup but
    keep the human-readable words.
    """
    if not text:
        return ''

    # --- 1. Remove {{template}} blocks ---
    # e.g. "{{Infobox|name=foo}}" or "{{convert|5|km}}" — these are
    # metadata/formatting instructions, not article text.
    # Run 3 times because templates can be nested one level inside another
    # (e.g. {{cite|date={{currentdate}}}}); each pass strips the outermost
    # surviving layer. A regex can't fully solve arbitrary nesting depth,
    # but article wikitext rarely nests more than 1-2 levels.
    for _ in range(3):
        text = re.sub(r'\{\{[^{}]*\}\}', '', text)

    # --- 2. Remove <ref>...</ref> citation blocks ---
    # e.g. "<ref>cited from XYZ, 2020</ref>" — citation text, not article prose.
    # DOTALL so '.' also matches newlines (refs can span multiple lines).
    text = re.sub(r'<ref[^>]*>.*?</ref>', '', text, flags=re.DOTALL)
    # Self-closing refs that just point to an earlier-defined ref: <ref name="x"/>
    text = re.sub(r'<ref[^>]*/>', '', text)

    # --- 3. Remove HTML comments ---
    # e.g. "<!-- editor note -->" — never meant to be visible text.
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)

    # --- 4. Remove any remaining raw HTML tags ---
    # e.g. "<b>", "<small>", "<div>" — formatting, strip the tags but
    # this step alone leaves the tag's inner text alone (we already
    # removed ref/comment *content* above; this just removes leftover tags).
    text = re.sub(r'<[^>]+>', '', text)

    # --- 5. Resolve wiki links, keep the display text ---
    # "[[Pahang|பகாங்கு]]" -> "பகாங்கு"  (piped link: show 2nd part)
    # "[[பகாங்கு]]"        -> "பகாங்கு"  (simple link: show as-is)
    # group(1) is the optional "Target|" part (discarded), group(2) is
    # what's actually displayed to a reader.
    text = re.sub(r'\[\[([^\]|]*\|)?([^\]]*)\]\]', r'\2', text)

    # --- 6. Resolve external links, keep the link's label text ---
    # "[https://example.com நன்றி]" -> "நன்றி"   (url + display text)
    text = re.sub(r'\[https?://\S+\s+([^\]]*)\]', r'\1', text)
    # "[https://example.com]" -> ""              (bare url, no label, just delete)
    text = re.sub(r'\[https?://\S+\]', '', text)

    # --- 7. Remove bold/italic markup ---
    # "'''bold'''" and "''italic''" are wikitext formatting, not characters
    # the reader actually sees as quotes — strip the quote-marks only.
    text = re.sub(r"'''|''", '', text)

    # --- 8. Remove section heading markers ---
    # "==Section Title==" -> "Section Title" (keep the heading text, drop the =)
    text = re.sub(r"^[=]+\s*|\s*[=]+$", '', text, flags=re.MULTILINE)

    return text

In [6]:
def parse_tamil_wiki(file_path, out_path, keep_ns=('0',)):
    """
    Stream-parse the (huge) XML dump page by page, clean each page's
    wikitext, keep only Tamil words, and write one line per page to a
    plain .txt file.
    """

    # iterparse = SAX-style streaming parser. It reads the file
    # incrementally instead of loading the whole 2GB tree into RAM at once
    # (which is what etree.parse() or pandas would try to do, and why
    # those choke on a file this size).
    #
    # tag='{*}page' -- your dump's root element declares a default XML
    # namespace (xmlns="http://www.mediawiki.org/xml/export-0.11/"), which
    # means every tag's *real* internal name is actually
    # "{http://www.mediawiki.org/xml/export-0.11/}page", not just "page".
    # The '{*}' wildcard tells lxml "match 'page' in ANY namespace" so we
    # don't have to hardcode that long namespace URL. This was the main
    # bug in your original snippet (it searched for tag='book' with no
    # namespace handling at all, so it would never match anything).
    ctx = etree.iterparse(file_path, events=('end',), tag='{*}page')

    n_pages, n_kept = 0, 0

    with open(out_path, 'w', encoding='utf-8') as out:
        # Loop fires once per </page> closing tag encountered in the file,
        # i.e. once the parser has read one full <page>...</page> block.
        for _, page in ctx:
            n_pages += 1

            # --- Filter to main article namespace only ---
            # MediaWiki dumps mix article pages (ns=0) with talk pages,
            # user pages, templates, etc (ns=1,2,3,4...). Your sample XML
            # had ns=4 (talk/project pages, full of chatter and
            # signatures) mixed in with ns=0 (actual articles). We only
            # want ns=0 for clean tokenizer training text.
            ns_el = page.find('{*}ns')
            if keep_ns is not None and (ns_el is None or ns_el.text not in keep_ns):
                page.clear()   # free memory even for skipped pages
                continue

            # --- Pull out the raw wikitext body ---
            # Structure is <page><revision><text>...wikitext...</text></revision></page>
            # We only grab the first <revision> here -- fine for this kind
            # of "current pages" dump, since each page only has 1 revision.
            text_el = page.find('{*}revision/{*}text')
            raw = text_el.text if text_el is not None and text_el.text else ''

            # --- Strip wikitext markup down to plain prose ---
            cleaned = clean_wikitext(raw)

            # --- Keep only words containing a Tamil character ---
            # We filter at the WORD level (split on whitespace), not by
            # deleting individual non-Tamil characters from the whole
            # blob. If we deleted char-by-char, an English word or stray
            # symbol sitting between two Tamil words would just vanish,
            # silently fusing the words on either side into one garbage
            # token. Filtering whole words avoids that.
            tamil_words = extract_tamil_words(cleaned)

            # Only write a line if the page actually had Tamil content
            # left after filtering (skips empty/stub pages).
            if tamil_words:
                out.write(' '.join(tamil_words) + '\n')
                n_kept += 1

            # --- Memory cleanup (same idea as your original code) ---
            # Without this, iterparse still keeps every processed element
            # around in the tree, and you'd slowly leak memory back up to
            # ~2GB+ as you scan the whole file.
            page.clear()
            while page.getprevious() is not None:
                del page.getparent()[0]

            if n_pages % 5000 == 0:
                print(f'{n_pages} pages scanned, {n_kept} kept')

    del ctx
    print(f'done: {n_pages} pages scanned, {n_kept} kept -> {out_path}')

In [7]:
# Store the tamil_wiki as a .txt file
cleaned_data = data / 'cleaned'
cleaned_data.mkdir(exist_ok=True, parents=True)

# Tamil wiki structured output
tamil_wiki_txt = cleaned_data / 'tamil_wiki.txt'

parse_tamil_wiki(out_path, tamil_wiki_txt)

10000 pages scanned, 7200 kept
15000 pages scanned, 11514 kept
20000 pages scanned, 14976 kept
25000 pages scanned, 18448 kept
35000 pages scanned, 25434 kept
40000 pages scanned, 29160 kept
50000 pages scanned, 35608 kept
60000 pages scanned, 42586 kept
65000 pages scanned, 46423 kept
70000 pages scanned, 50573 kept
75000 pages scanned, 54782 kept
80000 pages scanned, 58842 kept
85000 pages scanned, 62495 kept
95000 pages scanned, 69822 kept
110000 pages scanned, 80084 kept
115000 pages scanned, 83593 kept
120000 pages scanned, 86807 kept
130000 pages scanned, 94082 kept
145000 pages scanned, 106345 kept
150000 pages scanned, 111293 kept
160000 pages scanned, 118283 kept
165000 pages scanned, 121902 kept
170000 pages scanned, 125630 kept
175000 pages scanned, 129625 kept
180000 pages scanned, 134429 kept
185000 pages scanned, 138902 kept
190000 pages scanned, 143794 kept
195000 pages scanned, 148404 kept
200000 pages scanned, 151758 kept
205000 pages scanned, 155385 kept
210000 pages 

### Check quality of data cleaning

In [8]:
import os

with open(tamil_wiki_txt, encoding='utf-8') as f:
  for _ in range(10):
    print(f.readline())

விக்கிப்பீடியா மொழிகள்

2 முகலாயக் கட்டிடக்கலை முகலாய கட்டடக்கலையின் சிறந்த மற்றும் மிகவும் நுட்பமான எடுத்துக்காட்டாக விளங்கும் தாஜ் மஹால் கட்டடக்கலை என்பது கட்டடங்கள் மற்றும் அதன் உடல் கட்டமைப்புகளை வடிவமைத்தல் , செயல்முறைத் திட்டமிடல் , மற்றும் கட்டடங்கள் கட்டுவதை உள்ளடக்கியதாகும் கட்டடக்கலை படைப்புகள் , கட்டடங்கள் பொருள் வடிவம் , பெரும்பாலும் கலாச்சார சின்னங்களாக மற்றும் கலை படைப்புகளாக காணப்படுகின்றது வரலாற்று நாகரிகங்கள் பெரும்பாலும் அவர்களின் கட்டடக்கலை சாதனைகளின் மூலம் அடையாளம் காணப்படுகின்றன ஒரு விரிவான வரைவிலக்கணம் , பெருமட்டத்தில் , நகரத் திட்டமிடல் , நகர்ப்புற வடிவமைப்பு மற்றும் நிலத்தோற்றம் முதலியவற்றையும் , நுண்மட்டத்தில் , தளபாடங்கள் , உற்பத்திப்பொருள் முதலியவற்றை உள்ளடக்கிய , முழு உருவாக்கச் சூழலின் வடிவமைப்பைக் கட்டடக்கலைக்குள் அடக்கும் மேற்படி விடயத்தில் , தற்போது கிடைக்கும் மிகப் பழைய ஆக்கம் , பொ.ஊ முதலாம் நூற்றாண்டைச் சேர்ந்த உரோமானியக் கட்டடக் கலைஞரான விட்ருவியஸ் என்பாரது " கட்டடக்கலை தொடர்பில் " , என்ற நூலாகும் இவரது கூற்றுப்படி , நல்ல கட்டடம் , அழகு , உறுதி , பயன்

## CC-100 Monolingual Dataset from Web Crawl (Tamil)

In [9]:
! wget https://data.statmt.org/cc-100/ta.txt.xz

--2026-06-27 23:07:12--  https://data.statmt.org/cc-100/ta.txt.xz
Resolving data.statmt.org (data.statmt.org)... 129.215.32.28
Connecting to data.statmt.org (data.statmt.org)|129.215.32.28|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1382032064 (1.3G) [application/x-xz]
Saving to: ‘ta.txt.xz’

ta.txt.xz           100%[===================>]   1.29G  9.81MB/s    in 2m 20s  

2026-06-27 23:09:32 (9.43 MB/s) - ‘ta.txt.xz’ saved [1382032064/1382032064]



In [10]:
cc_100 = raw_data / Path('cc_100')
cc_100.mkdir(exist_ok=True, parents=True)
print('CC-100 folder Created...')

cc_100_txt = str(cc_100)+'cc_100.txt'
cc_100_txt

CC-100 folder Created...


'data/raw/cc_100cc_100.txt'

In [11]:
%%time

# Undo the compression
import lzma
import shutil

with lzma.open('ta.txt.xz', 'rt', encoding='utf-8') as f:
  with open(cc_100_txt, 'w') as txt:
    shutil.copyfileobj(f, txt)

print('CC-100 Decompression Complete...')

CC-100 Decompression Complete...
CPU times: user 3min 35s, sys: 13.1 s, total: 3min 48s
Wall time: 4min 1s


In [12]:
# Inspect the Raw cc_100 data
import numpy as np
import random

random_lines = random.sample(list(np.arange(0, 1000)), k=5)

with open(cc_100_txt, 'r') as f:
  for i in range(1000):
    if i in random_lines:
      print(f.readline())
    else:
      f.readline()

நாள் அங்க ஒரு நாள்

மேலதிக தகவல்கள் அடங்கிய துண்டுப்பிரசுரம் வெகுவிரைவில்...

பயணக்கட்டுரையை தொடர்ந்தும் எதிர்பார்க்கின்றோம்...

கட்சி செய்திகள்

இதில் முக்கியமானது பரோட்டாவை நன்கு 10 நிமிடம் வரை கொத்தினால் தான் நன்றாகவும் உதிரியாகவும் வரும்.



In [13]:
# Clean the text to remove links, headers, html formats and english words

def clean_cc100_line(line:str) -> list[str]:
  # Strip the URLs first
  text = re.sub(r'https?://\S+', '', line)
  # Strip HTML entities (&oldid=, &amp, etc)
  text = re.sub(r'&(?:[a-zA-Z]+|#[0-9]+|#x[0-9a-fA-F]+);', '', text)
  # Strip all English workds
  text = ENGLISH_WORD.sub('', text)
  # Extract out the tamil letters and number
  return TAMIL_RUN.findall(text)

### Clean the CC-100 data

In [14]:
%%time
from tqdm import tqdm

# Tamil CC-100 structured output
tamil_cc100 = cleaned_data / 'tamil_cc100.txt'

with open(cc_100_txt, 'r') as f_in, open(tamil_cc100, 'w') as f_out:
  for line in tqdm(f_in):
    tokens = clean_cc100_line(line)
    if tokens:
      f_out.write(' '.join(tokens) + '\n')

print('CC-100 text Cleaned...')

68237343it [15:28, 73513.27it/s]

CC-100 text Cleaned...
CPU times: user 14min 28s, sys: 34.1 s, total: 15min 2s
Wall time: 15min 28s


In [15]:
# Inspect the cleaned text.
random_lines = random.sample(list(np.arange(0, 10000)), k=5)

with open(tamil_cc100, 'r') as f:
  for i in range(10000):
    if i in random_lines:
      print(f.readline())
    else:
      f.readline()

அங்கு என்ற ஹிட்லர் காலத்தைய நிலையைக் காட்டும் அற்புத படம் இருந்தாலும் இது பற்றி கட்டாரில் இருக்கும் என் தம்பி அடிக்கடி புகழ்ந்து சொல்லிக் கொண்டே இருப்பான் எனக்காக அதை தம்பி இல் கொண்டு வந்திருப்பதனால் அதைப் பிறகு வீட்டிலேயே பார்க்கலாம் என விட்டு விட்டேன்

தொல்காப்பியச் சட்டம் என்ன சொல்கிறது என்று பாருங்கள்

ரூ 5 ஆயிரம் கோடி அபராதம் சேமிப்பு கணக்கில் மினிமம் பேலன்ஸ் இல்லாதவர்களிடமிருந்து வங்கிகள் வசூலிப்பு எஸ்பிஐ முதலிடம்

சந்தன வீரப்பன் பிடியில் சிக்கி காட்டில் பிணைக் கைதியாக இருக்கும் கன்னட நடிகர் ராஜ்குமாரை மீட்க தமிழகமும் , கர்நாடகமும்இணைந்து கூட்டு நடவடிக்கைகள் மேற்கொண்டுள்ளன இரு மாநில ஆதரவுடன் அரசு தூதராக நக்கீரன் கோபால் 4 முறைதனியாக காட்டுக்குச் சென்று வந்தார்

ம திரட்டியில் இருந்து மறுமொழிகள்



## Project Madurai

In [16]:
import requests
from bs4 import BeautifulSoup

response = requests.get('https://www.projectmadurai.org/pmworks.html')
soup = BeautifulSoup(response.text, 'html.parser')

html_links = {}

for a in soup.find_all('a', href=True):
  # Get all the urls present in the page
    href = a['href']
    if href.startswith('/pm_etexts/utf8/') and href.endswith('.html'):
      # Example : https://www.projectmadurai.org/pm_etexts/utf8/pmuni1081_02.html
        filename = a.text.strip()           # e.g. "pmuni0849.html"
        full_url = 'https://www.projectmadurai.org' + href
        html_links[filename] = full_url

In [17]:
print(f"Found {len(html_links)} HTML links")
print(list(html_links.items())[:5])  # preview first 5

Found 1358 HTML links
[('pmuni0001.html', 'https://www.projectmadurai.org/pm_etexts/utf8/pmuni0001.html'), ('pmuni0002.html', 'https://www.projectmadurai.org/pm_etexts/utf8/pmuni0002.html'), ('pmuni0003_01.html', 'https://www.projectmadurai.org/pm_etexts/utf8/pmuni0003_01.html'), ('pmuni0003_02.html', 'https://www.projectmadurai.org/pm_etexts/utf8/pmuni0003_02.html'), ('pmuni0004.html', 'https://www.projectmadurai.org/pm_etexts/utf8/pmuni0004.html')]


### Explore Data

In [18]:
sample = html_links['pmuni0001.html']

sample_response = requests.get(sample)
sample_response.encoding = 'utf-8'
soup = BeautifulSoup(sample_response.text, 'html.parser')
print(soup.body.get_text()[-500:])  # should show clean Tamil
print(soup.body.get_text()[:900])

அதுமன்னும்
கூடலிற் காணப் படும்.       1327
ஊடிப் பெறுகுவம் கொல்லோ நுதல்வெயர்ப்பக்
கூடலில் தோன்றிய உப்பு.       1328
ஊடுக மன்னோ ஒளியிழை யாமிரப்ப
நீடுக மன்னோ இரா.       1329
ஊடுதல் காமத்திற்கு இன்பம் அதற்கின்பம்
கூடி முயங்கப் பெறின்.       1330

கற்பியல் முற்றிற்று
காமத்துப்பால் முற்றிற்று
திருக்குறள் முற்றிற்று



This page was first put up on Sept 1, 2000 
This unicode version was last revised on 31 july 2021. 
Please send your comments and corrections to the Webmaster at pmadurai AT gmail.com


 


 tirukuRaL of tiruvaLLuvar:  (in Tamil Script, unicode/UTF-8 format) 
திருவள்ளுவர் அருளிய திருக்குறள்   


The Etext file was initially prepared in Mylai format during 1998 release. 
The content subsequently converted to Unicode encoding (utf-8 format). 
Preparation of HTML and PDF versions: Dr. K. Kalyanasundaram, Lausanne, Switzerland.

© Project Madurai 1998-2000
Project Madurai is an open, voluntary, worldwide initiative devoted to preparation of 
electronic texts of tamil literary work

In [19]:
sample = html_links['pmuni0002.html']

sample_response = requests.get(sample)
sample_response.encoding = 'utf-8'
soup = BeautifulSoup(sample_response.text, 'html.parser')
print(soup.body.get_text()[-500:])  # should show clean Tamil
print(soup.body.get_text()[:1000])

ுவமாம் சம்பறுத்தார் யாக்கைக்குப் 
 போனவா தேடும் பொருள்.       38

 முப்பதாம் ஆண்டளவில் மூன்றற்று ஒருபொருளைத் 
 தப்பாமல் தன்னுள் பெறானாயின் - செப்பும் 
 கலையளவே ஆகுமாம் காரிகையார் தங்கள் 
 முலையளவே ஆகுமாம் மூப்பு.        39

 தேவர் குறளும் திருநான் மறைமுடிவும் 
 மூவர் தமிழும் முனிமொழியும் - கோவை 
 திருவா சகமும் திருமூலர் சொல்லும் 
 ஒருவா சகமென் றுணர்.       40


This unicode version was last revised on 8 July 2021. 
Please send your comments and corrections to the webmaster (pmadurai@gmail.com) 

 


 Works of Auvaiyar :
AticuTi, konRai vEntan, mUturai  & nalvazi
(in Tamil Script, unicode/UTF-8 format) 
ஔவையார் நூல்கள்: ஆத்திச்சூடி, 
கொன்றைவேந்தன், மூதுரை  &  நல்வழி



Acknowledgements: 
Our Sincere thanks go to Dr. K. Kalyanasundaram, Lausanne, Switzerland for the preparation of this work for publication. 
This page was first put up on the Internet on June 21, 2001 
The work is presented in Unicode encoding (utf-8 format) 

© Project Madurai, 1998-2021.
Project Madurai is an open, volu

### Clean Project Madurai html data

In [20]:
# Clean the text to remove links, headers, html formats and english words

def clean_project_madurai_line(line:str) -> list[str]:
  # Strip the URLs first
  text = re.sub(r'https?://\S+', '', line)
  # Strip emails
  text = re.sub(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', '', text)
  # Strip HTML entities (&oldid=, &amp, etc)
  text = re.sub(r'&(?:[a-zA-Z]+|#[0-9]+|#x[0-9a-fA-F]+);', '', text)
  # Strip all English workds
  text = ENGLISH_WORD.sub('', text)
  # Extract out the tamil letters and number
  return TAMIL_RUN.findall(text)

In [21]:
lines = soup.body.get_text(separator='\n').splitlines()

cleaned_lines = []

for line in lines:
  clean_line = clean_project_madurai_line(line)
  if clean_line:
    cleaned_lines.append(' '.join(clean_line))

cleaned_lines[-300:]

['86 ஊக்கம் உடைமை ஆக்கத்திற்கு அழகு',
 '87 வெள்ளைக்கு இல்லை கள்ளச் சிந்தை',
 '88 வேந்தன் சீறின் ஆம் துணை இல்லை',
 '89 வைகல் தோறும் தெய்வம் தொழு',
 '90 ஒத்த இடத்து நித்திரை கொள்',
 '91 ஓதாதார்க்கு இல்லை உணர்வொடும் ஒழுக்கம்',
 '3 மூதுரை',
 'கடவுள் வாழ்த்து',
 'வாக்குண்டாம் நல்ல மனமுண்டாம் மாமலராள்',
 'நோக்குண்டாம் மேனி நுடங்காது - பூக்கொண்டு',
 'துப்பார் திருமேனி தும்பிக்கை யான்பாதம்',
 'தப்பாமல் சார்வார் தமக்கு',
 'நன்றி ஒருவர்க்குச் செய்தக்கால் அந்நன்றி',
 'என்று தருங்கோல் என வேண்டா - நின்று',
 'தளரா வளர்தெங்கு தாளுண்ட நீரைத்',
 'தலையாலே தான்தருத லால் 1',
 'நல்லார் ஒருவர்க்குச் செய்த உபகாரம்',
 'கல்மேல் எழுத்துப்போல் காணுமே - அல்லாத',
 'ஈரமிலா நெஞ்சத்தார்க் கீந்த உபகாரம்',
 'நீர் மேல் எழுத்துக்கு நேர் 2',
 'இன்னா இளமை வறுமைவந் தெய்தியக்கால்',
 'இன்னா அளவில் இனியவும் - இன்னாத',
 'நாளல்லா நாள்பூந்த நன்மலரும் போலுமே',
 'ஆளில்லா மங்கைக் கழகு 3',
 'அட்டாலும் பால் சுவையில் குன்றா து அளவளவாய்',
 'நட்டாலும் நண்பல்லார் நண்பல்லர்',
 'கெட்டாலும் மேன்மக்கள் மேன்மக்களே சங்கு',
 'சுட்டாலும் வெண்மை த

### Save formatted Data

1. Iterate through every html link in the dict
2. For each link get all the lines
    - Clean the line
    - Check if there is anything left
    - Add the line if there is something left to the resulting file
3. Save file in the cleaned folder

In [23]:
%%time

project_madurai = cleaned_data / Path('project_madurai.txt')

with open(project_madurai, 'w') as f:
  # 1. Iterate though the dicts
  for file_num, (file_name, file_url) in enumerate(html_links.items()):
    # 2.a request the page
    response = requests.get(file_url)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, 'html.parser')

    if not soup.body:
      continue
    # 2.b Get the lines in the page
    lines = soup.body.get_text(separator='\n').splitlines()

    # 2.c Iterate though the lines
    for line in lines:
      cleaned_line = clean_project_madurai_line(line)
      if cleaned_line:
        f.write(' '.join(cleaned_line) + '\n')

    if file_num % 50 == 0:
      print(f'Saved text upto file: {file_num}')

Saved text upto file: 0
Saved text upto file: 50
Saved text upto file: 100
Saved text upto file: 150
Saved text upto file: 200
Saved text upto file: 250
Saved text upto file: 300
Saved text upto file: 350
Saved text upto file: 400
Saved text upto file: 450
Saved text upto file: 500
Saved text upto file: 550
Saved text upto file: 600
Saved text upto file: 650
Saved text upto file: 700
Saved text upto file: 750
Saved text upto file: 800
Saved text upto file: 850
Saved text upto file: 900
Saved text upto file: 950
Saved text upto file: 1000
Saved text upto file: 1050
Saved text upto file: 1100
Saved text upto file: 1150
Saved text upto file: 1200
Saved text upto file: 1250
Saved text upto file: 1300
Saved text upto file: 1350
CPU times: user 2min 57s, sys: 1.92 s, total: 2min 59s
Wall time: 9min 17s


In [36]:
# Explore the formated data

random_lines = random.sample(list(np.arange(0,1000)), k=5)

with open(project_madurai, 'r') as f:
  for i in range(1000):
    if i in random_lines:
      print(f.readline())
    else:
      f.readline()

புறனழீஇப் பொய்த்து நகை 182

ஆவது போலக் கெடும் 283

இருள்நீங்கி இன்பம் பயக்கும் மருள்நீங்கி

மாடல்ல மற்றை யவை 400

இடிக்குந் துணையாரை யாள்வரை யாரே

